# Notebook 06 — Does the Telescope Actually Help?
## Post-training metrics, plots, and the with-vs-without comparison

---
### Story so far → the final test

- **NB 01–05:** we designed, built, and assembled the whole Telescope detector.
- **NB 06 (here):** the moment of truth. A trained model in hand, we measure whether the
  hyperbolic foveation lens actually improves detection — especially for the **distant**
  objects it was designed to rescue.

We answer three questions:
1. **How accurate is the model overall?** (standard detection metrics)
2. **Where does it help most?** (accuracy broken down by object distance)
3. **Is the lens worth it?** (the same model run *with* vs *without* foveation)

---
### Important: this notebook needs a trained model

The plots are only meaningful once you have trained Telescope (`python train.py ...`, see
the README). The next cell looks for your trained checkpoints.

- **If checkpoints are found** → we evaluate them and show your real results.
- **If not** → the notebook runs in **DEMO MODE**: every cell still executes so you can see
  exactly what the analysis will look like, but the live numbers come from an *untrained*
  model on synthetic data and are **not real accuracy**. Wherever that happens, a clear
  banner says so, and we show the paper's published numbers as the reference for what
  good results look like.

In [ ]:
import sys, pathlib, json, warnings
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from telescope import TelescopeModel
from telescope.eval import CocoEvaluator, DetectionResult
from telescope.box import riemannian_to_euclidean_box

torch.set_default_dtype(torch.float32)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {DEVICE}')

CLASS_NAMES = ['Car', 'Truck', 'Sign', 'Bike', 'Debris', 'Person', '__background__']
NUM_CLASSES = len(CLASS_NAMES)

---
## 1 · Configuration — point to your trained checkpoints

Edit these paths to match where `train.py` saved your runs. The defaults follow the
README's `compare` recipe:
- a **Telescope** run (foveation on), and
- a **baseline** run trained with `--no_foveation`.

If you only trained one model, leave the baseline path as-is; the notebook will instead
simulate the baseline by switching the lens off in the Telescope model at evaluation time.

In [ ]:
# ── Edit these to your paths ──────────────────────────────────────────────────
TELESCOPE_CKPT = pathlib.Path('../runs/telescope/checkpoint_best.pt')
BASELINE_CKPT  = pathlib.Path('../runs/baseline/checkpoint_best.pt')
VAL_DATA_DIR   = pathlib.Path('../data/argoverse2/sensor/val')

# ── Decide mode ───────────────────────────────────────────────────────────────
DEMO_MODE = not TELESCOPE_CKPT.exists()

if DEMO_MODE:
    print('=' * 70)
    print('  DEMO MODE — no trained checkpoint found at')
    print(f'    {TELESCOPE_CKPT}')
    print('  The notebook will run end-to-end on an UNTRAINED model + synthetic')
    print('  data so you can see the analysis format. Live numbers are NOT real')
    print('  accuracy. Train first (see README) then re-run for real results.')
    print('=' * 70)
else:
    print(f'Found Telescope checkpoint: {TELESCOPE_CKPT}')
    print(f'Baseline checkpoint exists : {BASELINE_CKPT.exists()}')

---
## 2 · Load the model(s)

We load the trained weights into a `TelescopeModel`. In demo mode we just build a fresh
(untrained) model so the rest of the notebook has something to run.

In [ ]:
def build_model():
    return TelescopeModel(num_classes=NUM_CLASSES, num_queries=300, query_dim=256).to(DEVICE)


def load_checkpoint(path):
    """Load a TelescopeModel from a checkpoint, or a fresh model in demo mode."""
    model = build_model()
    if path.exists():
        ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        epoch = ckpt.get('epoch', '?')
        print(f'Loaded {path.name}  (epoch {epoch})')
    else:
        print(f'(demo) using untrained model in place of {path.name}')
    model.eval()
    return model


if DEMO_MODE:
    # Use a smaller model in demo mode so it runs fast on CPU
    def build_model():
        return TelescopeModel(num_classes=NUM_CLASSES, num_queries=30, query_dim=64,
                              enc_out_dim=64).to(DEVICE)

telescope_model = load_checkpoint(TELESCOPE_CKPT)

# Baseline: either its own checkpoint, or we reuse the telescope model with the lens OFF
if BASELINE_CKPT.exists():
    baseline_model = load_checkpoint(BASELINE_CKPT)
    baseline_is_separate = True
else:
    baseline_model = telescope_model
    baseline_is_separate = False
    print('No separate baseline — will simulate it by disabling foveation at eval time.')

---
## 3 · Evaluation helpers

Two functions do the work:
- `run_inference` collects the model's detections over a dataset.
- `disable_foveation` lets us run the *same* model with the lens switched off (by forcing the
  radius R to ~0, which makes the warp Φ the identity — recall from NB 01 that a tiny R means
  no magnification). This is how we get an apples-to-apples 'without Telescope' comparison.

In [ ]:
from contextlib import contextmanager

@contextmanager
def disable_foveation(model):
    """Temporarily force the warp to identity (R->0) so the model runs WITHOUT the lens."""
    original_forward = model.warp_layer.forward
    def identity_warp(image, o, R):
        return image  # skip warping entirely
    model.warp_layer.forward = identity_warp
    try:
        yield model
    finally:
        model.warp_layer.forward = original_forward


@torch.no_grad()
def run_inference(model, loader, score_threshold=0.05, foveation=True):
    """Run the model over a dataset and return (predictions, ground_truth) lists.

    Each entry is a dict with boxes (cx,cy,w,h), labels, scores, and (for GT) areas.
    """
    preds, gts = [], []
    ctx = (lambda: disable_foveation(model)) if not foveation else _nullctx
    with ctx():
        for images, targets in loader:
            images = images.to(DEVICE)
            boxes_eu, logits, o, R = model(images)
            probs = logits.softmax(-1)[:, :, :-1]
            scores, labels = probs.max(-1)
            for b, tgt in enumerate(targets):
                keep = scores[b] > score_threshold
                preds.append(dict(
                    boxes=boxes_eu[b][keep].cpu(),
                    scores=scores[b][keep].cpu(),
                    labels=labels[b][keep].cpu(),
                ))
                gts.append(dict(
                    boxes=tgt['boxes'],
                    labels=tgt['labels'],
                ))
    return preds, gts


from contextlib import contextmanager as _cm
@_cm
def _nullctx():
    yield None

print('Evaluation helpers ready.')

---
## 4 · Get a validation set

In real mode we load the Argoverse 2 validation split. In demo mode we synthesise a small
set of images with known boxes at varying sizes (size stands in for distance: small box =
far away) so the distance-binned analysis below has something to chew on.

In [ ]:
def make_synthetic_loader(n_batches=6, B=2, device='cpu'):
    """Synthetic (image, targets) batches with boxes spanning near→far (large→small)."""
    torch.manual_seed(0)
    loader = []
    for _ in range(n_batches):
        images = torch.rand(B, 3, 128, 128)
        targets = []
        for b in range(B):
            n = torch.randint(3, 7, ()).item()
            boxes = torch.rand(n, 4)
            boxes[:, :2] = boxes[:, :2] * 1.2 - 0.6           # centres
            # sizes from large (near) to small (far)
            sizes = torch.linspace(0.25, 0.03, n)
            boxes[:, 2] = sizes
            boxes[:, 3] = sizes * 0.7
            targets.append(dict(
                boxes=boxes,
                labels=torch.randint(0, NUM_CLASSES - 1, (n,)),
                image_id=len(loader) * B + b,
            ))
        loader.append((images, targets))
    return loader


if DEMO_MODE:
    val_loader = make_synthetic_loader()
    print(f'Demo: synthesised {len(val_loader)} batches.')
else:
    from torch.utils.data import DataLoader
    from telescope.data import Argoverse2Dataset, collate_fn
    val_ds = Argoverse2Dataset(str(VAL_DATA_DIR), split='val', image_size=(1024, 1024))
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn,
                            num_workers=4)
    print(f'Loaded Argoverse2 val: {len(val_ds)} frames.')

---
## 5 · Overall accuracy — how to read it

We run the standard COCO detection metric, **mAP** (mean Average Precision). In one number,
mAP captures how well the predicted boxes match the true boxes across all classes and overlap
thresholds. **Higher is better; 1.0 is perfect, 0.0 is useless.**

We report it for the model **with** the lens and **without** it. If foveation helps, the
'with' bar should be taller.

In [ ]:
def coco_metrics(model, loader, foveation=True):
    ev = CocoEvaluator(num_classes=NUM_CLASSES - 1, class_names=CLASS_NAMES[:-1])
    ctx = _nullctx() if foveation else disable_foveation(model)
    with torch.no_grad(), ctx:
        for images, targets in loader:
            images = images.to(DEVICE)
            boxes_eu, logits, o, R = model(images)
            probs = logits.softmax(-1)[:, :, :-1]
            scores, labels = probs.max(-1)
            for b, tgt in enumerate(targets):
                keep = scores[b] > 0.05
                ev.update([DetectionResult(boxes_eu[b][keep].cpu(), scores[b][keep].cpu(),
                                            labels[b][keep].cpu(), tgt['image_id'])], [tgt])
    return ev.summarize()


with_lens    = coco_metrics(telescope_model, val_loader, foveation=True)
without_lens = coco_metrics(
    baseline_model, val_loader,
    foveation=baseline_is_separate   # if separate baseline, use it as-is; else disable lens
)

print('\nWith foveation   :', {k: round(v, 4) for k, v in with_lens.items()})
print('Without foveation:', {k: round(v, 4) for k, v in without_lens.items()})

if DEMO_MODE:
    print('\n[DEMO] Numbers above are from an untrained model — not real accuracy.')

In [ ]:
keys = ['mAP_50', 'mAP']
labels_pretty = ['mAP@50  (lenient overlap)', 'mAP  (strict, COCO avg)']
with_vals    = [with_lens.get(k, 0) for k in keys]
without_vals = [without_lens.get(k, 0) for k in keys]

x = np.arange(len(keys)); w = 0.35
fig, ax = plt.subplots(figsize=(7, 4.5))
b1 = ax.bar(x - w/2, without_vals, w, label='Without lens (baseline)', color='#888888')
b2 = ax.bar(x + w/2, with_vals,    w, label='With lens (Telescope)',   color='#4C72B0')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels_pretty)
ax.set_ylabel('mAP  (higher = better)')
ax.set_title(('[DEMO — untrained] ' if DEMO_MODE else '') + 'Overall accuracy: with vs without the lens')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

print('How to read: taller blue than grey = foveation improves accuracy.')

---
## 6 · The key result — accuracy by distance

This is the plot the whole paper is about. Telescope's claim is not 'better everywhere' — it's
'**much** better on **far** objects, with no real cost up close'. So we split the objects into
distance bands and measure accuracy in each.

Since true depth isn't in the box labels, we use **box size as a stand-in for distance**:
a large box is a near object, a small box is a far one (a car 500 m away really is just a few
pixels). We bucket objects into Near / Mid / Far / Ultra-far by size and compute accuracy in each.

**How to read the plot:** the x-axis is distance band (near → ultra-far); the y-axis is
accuracy. Watch the **gap between the two lines grow** as you move right: that widening gap is
the telescope earning its keep exactly where ordinary detectors fail.

In [ ]:
def iou_xywh(a, b):
    ax1, ay1 = a[0]-a[2]/2, a[1]-a[3]/2; ax2, ay2 = a[0]+a[2]/2, a[1]+a[3]/2
    bx1, by1 = b[0]-b[2]/2, b[1]-b[3]/2; bx2, by2 = b[0]+b[2]/2, b[1]+b[3]/2
    ix1, iy1 = max(ax1,bx1), max(ay1,by1); ix2, iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter/union if union > 0 else 0.0


def recall_by_distance(preds, gts, bins, iou_thr=0.5):
    """Fraction of GT boxes correctly detected, bucketed by box size (distance proxy).

    bins: list of (name, min_area, max_area). Returns {name: recall}.
    """
    hits = {name: 0 for name, _, _ in bins}
    total = {name: 0 for name, _, _ in bins}
    for pred, gt in zip(preds, gts):
        gt_boxes = gt['boxes']
        pred_boxes = pred['boxes']
        for gi in range(len(gt_boxes)):
            area = float(gt_boxes[gi, 2] * gt_boxes[gi, 3])
            band = next((nm for nm, lo, hi in bins if lo <= area < hi), None)
            if band is None:
                continue
            total[band] += 1
            # is there a predicted box overlapping this GT?
            matched = any(iou_xywh(gt_boxes[gi].tolist(), pred_boxes[pi].tolist()) >= iou_thr
                          for pi in range(len(pred_boxes)))
            hits[band] += int(matched)
    return {nm: (hits[nm] / total[nm] if total[nm] else 0.0) for nm, _, _ in bins}


# Distance bands by box area (large area = near, small area = far)
BINS = [
    ('Near',      0.02,  1.00),
    ('Mid',       0.008, 0.02),
    ('Far',       0.003, 0.008),
    ('Ultra-far', 0.0,   0.003),
]

preds_with,    gts = run_inference(telescope_model, val_loader, foveation=True)
preds_without, _   = run_inference(baseline_model,  val_loader,
                                    foveation=baseline_is_separate)

rec_with    = recall_by_distance(preds_with,    gts, BINS)
rec_without = recall_by_distance(preds_without, gts, BINS)

band_names = [b[0] for b in BINS]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(band_names, [rec_without[n] for n in band_names], 'o--', color='#888888',
        linewidth=2, markersize=8, label='Without lens (baseline)')
ax.plot(band_names, [rec_with[n] for n in band_names], 'o-', color='#4C72B0',
        linewidth=2, markersize=8, label='With lens (Telescope)')
ax.set_xlabel('Distance band  (near → ultra-far)')
ax.set_ylabel('Recall @ IoU 0.5  (fraction of objects found)')
ax.set_title(('[DEMO — untrained] ' if DEMO_MODE else '') +
             'Detection accuracy vs distance')
ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(-0.02, 1.02)
plt.tight_layout(); plt.show()

if DEMO_MODE:
    print('[DEMO] Untrained model — expect near-zero, noisy bars. The SHAPE to expect after')
    print('       training is shown in the paper reference plot below.')

---
## 7 · Reference: what the paper reports

So you know what a *successful* result looks like, here are the published numbers from the
Telescope paper (Table 4, TruckDrive dataset). These are the authors' measured results, not
ours — shown as the target your trained model should approach.

Notice how the advantage **grows with distance**: roughly +0.25 mAP at the 250m+ band, a 76%
relative jump over the baseline. That widening gap is the whole thesis of the paper.

In [ ]:
# Published numbers from Telescope paper, Table 4 (mAP by distance bin)
paper_bins   = ['0-50m', '50-150m', '150-250m', '250m+']
paper_deform = [0.396, 0.178, 0.081, 0.072]   # Deformable DETR baseline (mAP_50-ish row)
paper_tele   = [0.608, 0.507, 0.335, 0.326]   # Telescope

x = np.arange(len(paper_bins)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar(x - w/2, paper_deform, w, label='Baseline (Deformable DETR)', color='#888888')
b2 = ax.bar(x + w/2, paper_tele,   w, label='Telescope', color='#55A868')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
# annotate the growing gap
for i in range(len(paper_bins)):
    gap = paper_tele[i] - paper_deform[i]
    ax.annotate(f'+{gap:.2f}', (x[i], max(paper_tele[i], paper_deform[i]) + 0.04),
                ha='center', fontsize=9, color='#2A6F3B', fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(paper_bins)
ax.set_xlabel('Object distance'); ax.set_ylabel('mAP')
ax.set_title('PAPER REFERENCE — Telescope vs baseline by distance (TruckDrive, Table 4)')
ax.legend(); ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 0.72)
plt.tight_layout(); plt.show()

print('The +numbers above each pair show the accuracy gain. It grows with distance —')
print('that is the telescope effect: it rescues the far objects ordinary detectors miss.')

---
## 8 · Qualitative comparison — seeing the detections

Numbers are convincing but pictures are intuitive. Here we draw the actual detections on a
few images, side by side: **without** the lens (left) vs **with** it (right). Green boxes are
the model's detections; dashed red boxes are the ground truth.

**What to look for:** on the 'with lens' side, more of the small/distant ground-truth boxes
should have a matching green detection. That is the visual signature of the foveation helping.

In [ ]:
@torch.no_grad()
def detect_on_image(model, image, foveation=True, score_threshold=0.2):
    ctx = _nullctx() if foveation else disable_foveation(model)
    with ctx:
        boxes_eu, logits, o, R = model(image.unsqueeze(0).to(DEVICE))
    probs = logits[0].softmax(-1)[:, :-1]
    scores, labels = probs.max(-1)
    keep = scores > score_threshold
    return boxes_eu[0][keep].cpu(), labels[keep].cpu(), scores[keep].cpu()


def draw(ax, image, det_boxes, gt_boxes, title):
    img = image.permute(1, 2, 0).numpy()
    ax.imshow(img); ax.set_title(title, fontsize=10); ax.axis('off')
    H, W = image.shape[-2:]
    def to_px(b):
        cx, cy, w, h = b
        return ((cx+1)/2*W - (w/2)*W, (cy+1)/2*H - (h/2)*H, w*W, h*H)
    for b in gt_boxes:
        x, y, w, h = to_px(b.tolist())
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False,
                     edgecolor='red', linestyle='--', linewidth=1.5))
    for b in det_boxes:
        x, y, w, h = to_px(b.tolist())
        ax.add_patch(patches.Rectangle((x, y), w, h, fill=False,
                     edgecolor='lime', linewidth=1.5))


# Take 3 sample images from the first batch
sample_images, sample_targets = val_loader[0] if isinstance(val_loader, list) else next(iter(val_loader))
n_show = min(3, len(sample_images))

fig, axes = plt.subplots(n_show, 2, figsize=(9, 4.2 * n_show))
if n_show == 1:
    axes = axes[None, :]
for i in range(n_show):
    img = sample_images[i]
    gt  = sample_targets[i]['boxes']
    db_off, _, _ = detect_on_image(baseline_model, img, foveation=baseline_is_separate)
    db_on,  _, _ = detect_on_image(telescope_model, img, foveation=True)
    draw(axes[i, 0], img, db_off, gt, 'Without lens' + (' [demo]' if DEMO_MODE else ''))
    draw(axes[i, 1], img, db_on,  gt, 'With lens (Telescope)' + (' [demo]' if DEMO_MODE else ''))

fig.suptitle('Detections: green = predicted, dashed red = ground truth', fontsize=11)
plt.tight_layout(); plt.show()

if DEMO_MODE:
    print('[DEMO] Untrained model places boxes randomly. After training, the right column')
    print('       should catch more of the small dashed-red distant boxes than the left.')

---
## 9 · The verdict

The cell below prints a compact summary table comparing with vs without the lens, and states
the headline conclusion. Run it after training for your real verdict.

In [ ]:
print('=' * 60)
print('  TELESCOPE — RESULTS SUMMARY')
print('=' * 60)
if DEMO_MODE:
    print('  MODE: DEMO (untrained) — numbers below are placeholders.')
    print('  Train with `python train.py ...` then re-run this notebook.')
else:
    print('  MODE: trained checkpoints')
print('-' * 60)
print(f'  {"Metric":<22}{"Without":>10}{"With":>10}{"Delta":>10}')
for k in ['mAP_50', 'mAP']:
    a = without_lens.get(k, 0); b = with_lens.get(k, 0)
    print(f'  {k:<22}{a:>10.3f}{b:>10.3f}{b-a:>+10.3f}')
print('-' * 60)
print('  Recall by distance (with lens):')
for nm in band_names:
    print(f'    {nm:<12}{rec_with[nm]:>6.2f}   (baseline {rec_without[nm]:.2f})')
print('=' * 60)

if not DEMO_MODE:
    far_gain = rec_with['Ultra-far'] - rec_without['Ultra-far']
    if far_gain > 0.02:
        print(f'  VERDICT: foveation improves ultra-far recall by +{far_gain:.2f}. The lens helps.')
    else:
        print('  VERDICT: little far-range gain here — check training length / params.')

---
## That's the whole journey

From a few lines of hyperbolic geometry in NB 01 to a complete, trained, evaluated
ultra-long-range detector. To recap the thread:

1. **NB 01** — ground the lens (the warp maths).
2. **NB 02** — mount it on images.
3. **NB 03** — tell the detector how it's set.
4. **NB 04** — detect through it and un-warp the results.
5. **NB 05** — assemble and train the full instrument.
6. **NB 06** — confirm it sees farther.

If your trained numbers show the with-lens line pulling away from the baseline at distance,
you have reproduced the paper's core finding: **explicitly magnifying the far field, in a
fully learnable and invertible way, rescues the distant objects that ordinary detectors miss.**